# Week 1 Exercise

This is a tool that takes a technical question, and responds with an explanation using gpt-5-nano.

In [31]:
# imports
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown, update_display

In [32]:
# constants
MODEL_GPT = 'gpt-5-nano'

In [33]:
# set up environment
load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("❌ No API key found - make sure OPENAI_API_KEY is set in your .env file")
else:
    print("✅ API key found!")
    client = OpenAI(api_key=api_key)

✅ API key found!


In [34]:
# here is the question; type over this to ask something new

system_prompt = (
    "You are a helpful technical assistant. "
    "When given a technical or coding question, provide a clear, "
    "concise explanation suitable for a software developer."
)

question = """
What is the difference between a token and a word in the context of LLMs?
"""

In [35]:
# Get gpt-5-nano to answer, with streaming

stream = client.chat.completions.create(
    model=MODEL_GPT,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": question}
    ],
    stream=True
)

response = ""
display_handle = display(Markdown(""), display_id=True)

for chunk in stream:
    delta = chunk.choices[0].delta.content or ""
    response += delta
    update_display(Markdown(response), display_id=display_handle.display_id)


Short answer:
- A word is a human-language unit (like "running" or "unbelievable").
- A token is a unit the model actually processes, created by the model’s tokenizer. Tokens can be whole words, subwords, characters, punctuation, or special tokens, depending on the tokenizer.

Details:
- Tokenization is the process of breaking text into tokens. For many LLMs, words are split into smaller pieces (subwords) to keep the vocabulary size reasonable and to handle new or rare words gracefully.
- Because of subword tokenization, a single word may become multiple tokens. For example, depending on the tokenizer, "unbelievable" might be split into pieces like ["un", "believ", "able"] or similar. Conversely, common words might map to a single token.
- Tokens include punctuation and spaces in some tokenizers (or they’re represented via special tokens). There are also special tokens like start/end-of-sequence tokens that don’t correspond to normal words.
- The model’s context window (the number of tokens it can read and generations it can produce) is in tokens, not words. This means counting tokens is what affects cost and maximum input length.

Examples:
- Word: "Hello, world!" (two words plus punctuation)
  - Tokens (depends on tokenizer): could be ["Hello", ",", "Ġworld", "!"] in some GPT-2-like tokenizers, or ["Hello", ",", "world", "!"] in others.
- Word with subword split: "tokenization" might become ["tok", "en", "ization"] in a subword tokenizer.

Practical takeaway:
- Treat tokens as the model’s processing units, not the human-friendly words.
- For estimating input length, cost, or limits, count tokens rather than words.
- Tokenization schemes vary by model (WordPiece, BPE, SentencePiece, etc.), so the exact tokenization of a word depends on the model you’re using. If you want to see how a specific text is tokenized, try the model’s tokenizer (e.g., via Hugging Face) to print the token list.